# 🌲 Python Segment Tree / BIT — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A Segment Tree is like a corporate org chart for an array. The CEO node knows the total for the whole company. Each VP covers half the employees. Each manager covers a quarter. To ask "what's the sum for region [l..r]?" you only visit the managers whose regions fully fit inside yours — never more than O(log n) stops. To update one employee, you visit them and update every manager above them — again O(log n). A BIT (Fenwick Tree) is a cleverer, flatter version that stores partial overlapping sums using bit tricks — faster to code, same asymptotic cost.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Segment Tree / BIT? The Visual Model](#1) |
| 2 | [Creating / Setup — Segment Tree and BIT Classes](#2) |
| 3 | [The Core API — Query, Update](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Segment Tree — Build, Query, Update](#5) |
| 6 | [Pattern 2: Binary Indexed Tree (BIT / Fenwick)](#6) |
| 7 | [Pattern 3: Range Sum Query Mutable (LC 307)](#7) |
| 8 | [Pattern 4: Count of Smaller Numbers After Self (LC 315)](#8) |
| 9 | [Pattern 5: Count of Range Sum (LC 327)](#9) |
| 10 | [The Segment Tree / BIT Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. 🗺️ What Is Segment Tree / BIT? The Visual Model

```
SEGMENT TREE on nums = [3, 1, 2, 7]

                    node 1
                  sum[0..3]=13
                 /              \
           node 2                node 3
         sum[0..1]=4           sum[2..3]=9
          /       \             /         \
       node4      node5      node6       node7
       [0]=3      [1]=1      [2]=2       [3]=7

Array storage (1-indexed, tree[0] unused):
  idx:  1    2    3    4    5    6    7
  val: 13    4    9    3    1    2    7

Node i →  left child: 2*i   right child: 2*i+1   parent: i//2

QUERY sum[1..3]:  visit node1? partial → visit node2(full) + node3(full)
                  total visits: O(log n)

UPDATE index 1, val→5:  node5=5 → node2=3+5=8 → node1=8+9=17
                        path length: O(log n)

─────────────────────────────────────────────────────────────────
BINARY INDEXED TREE (BIT / Fenwick) on same array:

  idx:    1    2    3    4
  bit:    3    4    2   13
          │    │    │    │
         [1] [1,2] [3] [1,4]  ← range each BIT cell is responsible for

  lowbit(i) = i & (-i)  → lowest set bit, tells you the range width

  Query prefix[1..4]: i=4 → +bit[4]=13  → done (lowbit(4)=4, covers all)
  Query prefix[1..3]: i=3 → +bit[3]=2, i=2 → +bit[2]=4, i=0 → done = 6 ✓
  Update index 2: i=2 → bit[2]+=, i=4 → bit[4]+=  (walk up via i+=lowbit(i))
─────────────────────────────────────────────────────────────────
```

<a id='2'></a>
## 2. 🔧 Creating / Setup — Segment Tree and BIT Classes

In [ ]:
# SEGMENT TREE — array-based, 1-indexed, supports range sum and point update

class SegmentTree:
    def __init__(self, nums):
        self.n = len(nums)
        self.tree = [0] * (4 * self.n)   # 4n is a safe upper bound for node count
        if nums:
            self._build(nums, 1, 0, self.n - 1)

    def _build(self, nums, node, l, r):
        if l == r:
            self.tree[node] = nums[l]     # leaf: store the element directly
            return
        mid = (l + r) // 2
        self._build(nums, 2 * node, l, mid)         # build left subtree
        self._build(nums, 2 * node + 1, mid + 1, r) # build right subtree
        self.tree[node] = self.tree[2*node] + self.tree[2*node+1]  # merge up

    def update(self, idx, val):
        self._update(1, 0, self.n - 1, idx, val)

    def _update(self, node, l, r, idx, val):
        if l == r:
            self.tree[node] = val         # found the leaf — set new value
            return
        mid = (l + r) // 2
        if idx <= mid:
            self._update(2 * node, l, mid, idx, val)
        else:
            self._update(2 * node + 1, mid + 1, r, idx, val)
        self.tree[node] = self.tree[2*node] + self.tree[2*node+1]  # recompute parent

    def query(self, ql, qr):
        return self._query(1, 0, self.n - 1, ql, qr)

    def _query(self, node, l, r, ql, qr):
        if ql <= l and r <= qr:
            return self.tree[node]        # full overlap — return this node's sum
        if qr < l or r < ql:
            return 0                      # no overlap — contribute nothing
        mid = (l + r) // 2
        return (self._query(2*node, l, mid, ql, qr) +
                self._query(2*node+1, mid+1, r, ql, qr))  # partial overlap


# BINARY INDEXED TREE (BIT / Fenwick) — 1-indexed, prefix sum + point update

class BIT:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)   # tree[0] unused; elements at 1..n

    def update(self, i, delta):
        while i <= self.n:
            self.tree[i] += delta
            i += i & (-i)           # walk UP: add lowest set bit → next responsible range

    def query(self, i):
        total = 0
        while i > 0:
            total += self.tree[i]
            i -= i & (-i)           # walk DOWN: strip lowest set bit → previous range
        return total

    def range_query(self, l, r):    # both 1-indexed, inclusive
        return self.query(r) - self.query(l - 1)


# Demo both on nums = [3, 1, 2, 7]
nums = [3, 1, 2, 7]
st = SegmentTree(nums)
bit = BIT(len(nums))
for i, v in enumerate(nums):
    bit.update(i + 1, v)            # BIT is 1-indexed

print("Segment Tree range sum [0..3]:", st.query(0, 3))   # 13
print("Segment Tree range sum [1..2]:", st.query(1, 2))   # 3
print("BIT prefix sum [1..4]:", bit.query(4))              # 13
print("BIT range sum [2..3]:", bit.range_query(2, 3))      # 3
st.update(1, 5)                                            # set index 1 to 5
print("After update index1=5, ST sum [0..3]:", st.query(0, 3))  # 17
bit.update(2, 4)                                           # BIT: add 4 to index 2
print("After BIT update +4 at 2, prefix[1..4]:", bit.query(4))  # 17

<a id='3'></a>
## 3. ⚡ The Core API — Query, Update

```
SEGMENT TREE
OPERATION                     COMPLEXITY   WHAT IT DOES
──────────────────────────────────────────────────────────────────
SegmentTree(nums)              O(n)         build tree from array
st.query(ql, qr)               O(log n)     sum of nums[ql..qr] (0-indexed)
st.update(idx, val)            O(log n)     set nums[idx] = val, recompute
──────────────────────────────────────────────────────────────────

BIT (FENWICK TREE)
OPERATION                     COMPLEXITY   WHAT IT DOES
──────────────────────────────────────────────────────────────────
BIT(n)                         O(1)         empty tree for n elements (1-indexed)
bit.update(i, delta)           O(log n)     add delta to position i, propagate up
bit.query(i)                   O(log n)     prefix sum from 1 to i (inclusive)
bit.range_query(l, r)          O(log n)     sum from l to r = query(r)-query(l-1)
i & (-i)  →  lowbit trick      O(1)         isolate lowest set bit (range width)
──────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  BIT with 0-indexed i=0 — lowbit(0)=0 → infinite loop; BIT is always 1-indexed
✅  Shift input: rank[v] = idx + 1, query(rank - 1) for strictly less
❌  Use segment tree for prefix-sum-only problems — BIT is simpler and faster
✅  Use segment tree when you need range min/max or lazy propagation
❌  Allocate tree[2*n] — use tree[4*n] for safety (tree may need up to 4n nodes)
```

In [ ]:
# LIVE DEMO: the lowbit trick — why BIT works

def lowbit(i):
    return i & (-i)    # isolates the lowest set bit

print("lowbit demo:")
for i in range(1, 9):
    lb = lowbit(i)
    print(f"  i={i:2d}  binary={bin(i):>10}  lowbit={lb}  responsible for range of size {lb}")

print()
print("BIT update path: updating index 3 in BIT of size 8")
i = 3
while i <= 8:
    print(f"  update bit[{i}]  ({bin(i)})  →  next: i += lowbit({i}) = {i + lowbit(i)}")
    i += lowbit(i)

print()
print("BIT query path: prefix sum 1..6")
i = 6
path = []
while i > 0:
    path.append(i)
    print(f"  read bit[{i}]  ({bin(i)})  →  next: i -= lowbit({i}) = {i - lowbit(i)}")
    i -= lowbit(i)
print(f"  sum = bit[{path[0]}] + bit[{path[1]}] + bit[{path[2]}]  =  prefix[1..6]")

<a id='4'></a>
## 4. 🗂️ Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                         WHAT TO USE
──────────────────────────────────────────────────────────────────
"range sum + point update" (immutable range)  BIT (simpler, faster to code)
"range sum + range update"                    Segment Tree with lazy propagation
"range min / max query"                       Segment Tree (BIT can't do min/max)
"count elements < x processed so far"         BIT + coordinate compression
"count range sums in [lower, upper]"           Merge sort divide-and-conquer
──────────────────────────────────────────────────────────────────

BIT vs SEGMENT TREE:
  BIT    → simpler code, O(log n) prefix sums only, 1-indexed
  SegTree → range min/max, lazy range updates, harder to implement

COORDINATE COMPRESSION PATTERN:
  Problem: BIT indexed by value, but values can be huge (up to 10^9)
  Fix:     compress values to ranks [1..m] where m = number of distinct values
    sorted_unique = sorted(set(nums))
    rank = {v: i+1 for i, v in enumerate(sorted_unique)}
  Now BIT size = m instead of max_value

MERGE SORT COUNT PATTERN:
  When counting inversions or range sums: sort while counting
  During merge: left half sorted, right half sorted → cross-half pairs easy to count
  Two pointers walk right half counting how many left values are in valid range
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Segment Tree — Build, Query, Update

---

```
PROBLEM:  Build a structure on nums supporting O(log n) range sum and point update.
APPROACH: 1-indexed array tree. Node i covers range [l..r].
          Left child: 2i covers [l..mid]. Right child: 2i+1 covers [mid+1..r].
          Tree needs 4n space (worst case for non-power-of-2 n).

SLOW MOTION BUILD on nums = [1, 3, 5, 7, 9, 11]:

  _build(node=1, l=0, r=5)  → sum = 36
    _build(node=2, l=0, r=2)  → sum = 9
      _build(node=4, l=0, r=1)  → sum = 4
        _build(node=8, l=0, r=0)  → leaf: tree[8]=1
        _build(node=9, l=1, r=1)  → leaf: tree[9]=3
        tree[4] = tree[8]+tree[9] = 4
      _build(node=5, l=2, r=2)  → leaf: tree[5]=5
      tree[2] = tree[4]+tree[5] = 9
    _build(node=3, l=3, r=5)  → sum = 27  (similar)
    tree[1] = 9 + 27 = 36

SLOW MOTION QUERY sum[1..4] on built tree:
  _query(1, 0,5, 1,4): partial → recurse both children
    _query(2, 0,2, 1,4): partial → recurse
      _query(4, 0,1, 1,4): full overlap [0..1] inside [1..4]? NO (0<1)
                           partial → recurse
        _query(8, 0,0, 1,4): 0 < 1 → no overlap → return 0
        _query(9, 1,1, 1,4): full → return tree[9]=3
      _query(5, 2,2, 1,4): full → return tree[5]=5
    _query(3, 3,5, 1,4): partial → recurse
      _query(6, 3,4, 1,4): full → return tree[6]=16
      _query(7, 5,5, 1,4): 5 > 4 → no overlap → return 0
  total = 0+3+5+16+0 = 24  ✓  (3+5+7+9=24)

KEY INSIGHT: Full overlap → return immediately without recursing deeper.
             This is what makes query O(log n) not O(n).
TIME:  O(n) build, O(log n) query and update
SPACE: O(n) — tree array of size 4n
```

In [ ]:
class SegTreeFull:
    """
    Segment Tree (array-based, 1-indexed, range sum, point update).
    Approach: recursive build from center; full/no overlap base cases in query.
    Time:  O(n) build, O(log n) query and update
    Space: O(n) — tree array of 4n nodes
    """
    def __init__(self, nums):
        self.n = len(nums)
        self.tree = [0] * (4 * self.n)   # safe upper bound
        self._build(nums, 1, 0, self.n - 1)

    def _build(self, nums, node, l, r):
        if l == r:
            self.tree[node] = nums[l]      # leaf: exact element
            return
        mid = (l + r) // 2
        self._build(nums, 2*node, l, mid)
        self._build(nums, 2*node+1, mid+1, r)
        self.tree[node] = self.tree[2*node] + self.tree[2*node+1]  # bottom-up merge

    def update(self, idx, val):
        """Set nums[idx] = val. O(log n)."""
        self._update(1, 0, self.n - 1, idx, val)

    def _update(self, node, l, r, idx, val):
        if l == r:
            self.tree[node] = val          # found leaf — overwrite
            return
        mid = (l + r) // 2
        if idx <= mid:
            self._update(2*node, l, mid, idx, val)
        else:
            self._update(2*node+1, mid+1, r, idx, val)
        self.tree[node] = self.tree[2*node] + self.tree[2*node+1]  # recompute after update

    def query(self, ql, qr):
        """Sum of nums[ql..qr] inclusive, 0-indexed. O(log n)."""
        return self._query(1, 0, self.n - 1, ql, qr)

    def _query(self, node, l, r, ql, qr):
        if ql <= l and r <= qr:
            return self.tree[node]         # full overlap — return without recursing
        if qr < l or r < ql:
            return 0                       # no overlap — contribute nothing
        mid = (l + r) // 2
        return (self._query(2*node, l, mid, ql, qr) +
                self._query(2*node+1, mid+1, r, ql, qr))

# Slow motion on nums=[1,3,5,7,9,11]:
# query(1,4) → visits O(log n) nodes, returns 3+5+7+9=24
# update(2,10) → sets nums[2]=10, recomputes path from leaf to root

def test_harness(st_class):
    tests = [
        ([1, 3, 5, 7, 9, 11], 0, 5, 36),     # full range
        ([1, 3, 5, 7, 9, 11], 1, 4, 24),     # partial range: 3+5+7+9
        ([1, 3, 5, 7, 9, 11], 2, 2, 5),      # single element
        ([2, 4], 0, 1, 6),                   # two elements
    ]
    passed = 0
    for nums, ql, qr, expected in tests:
        st = st_class(nums)
        got = st.query(ql, qr)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | nums={nums} query[{ql}..{qr}] | expected={expected} | got={got}")
        passed += (got == expected)

    # update test
    st = st_class([1, 3, 5, 7])
    st.update(1, 10)                          # set index 1 to 10
    got = st.query(0, 3)                      # should be 1+10+5+7=23
    ok = got == 23
    passed += ok
    if not ok:
        print(f"FAILED | update test | expected=23 | got={got}")
    print(f"{passed}/{len(tests)+1} tests passed")

test_harness(SegTreeFull)
print("SegTreeFull defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Binary Indexed Tree (BIT / Fenwick)

---

```
PROBLEM:  Build a structure on n elements supporting O(log n) prefix sum and point update.
APPROACH: BIT stores partial sums. Each index i is responsible for a range of size lowbit(i).
          lowbit(i) = i & (-i)  →  lowest set bit of i.

LOWBIT DEMO:
  i=1  binary=001  lowbit=1  → responsible for range [1..1]
  i=2  binary=010  lowbit=2  → responsible for range [1..2]
  i=3  binary=011  lowbit=1  → responsible for range [3..3]
  i=4  binary=100  lowbit=4  → responsible for range [1..4]
  i=6  binary=110  lowbit=2  → responsible for range [5..6]

SLOW MOTION UPDATE index 3, delta=+5:
  i=3  → bit[3]+=5  i += lowbit(3)=1  → i=4
  i=4  → bit[4]+=5  i += lowbit(4)=4  → i=8 (>n, stop)
  Updated: bit[3] and bit[4]  (both cover range containing index 3)

SLOW MOTION QUERY prefix sum 1..5:
  i=5  → sum+=bit[5]  i -= lowbit(5)=1  → i=4
  i=4  → sum+=bit[4]  i -= lowbit(4)=4  → i=0 (stop)
  sum = bit[5] + bit[4] = covers [5..5] + [1..4] = prefix[1..5] ✓

KEY INSIGHT: The bit-manipulation ensures every prefix query hits exactly
             the minimal set of non-overlapping ranges that tile [1..i].
             Update propagates to all ranges that contain index i.
TIME:  O(n) to build from scratch (n updates), O(log n) per update/query
SPACE: O(n) — single flat array of size n+1
```

In [ ]:
class BITFull:
    """
    Binary Indexed Tree (Fenwick Tree) — prefix sum + point update.
    Approach: lowbit trick routes updates/queries to exactly the right cells.
    IMPORTANT: 1-indexed. Pass i in range [1..n]. Never pass i=0.
    Time:  O(log n) per update and query
    Space: O(n) — single flat array
    """
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)   # tree[0] is unused — BIT is 1-indexed

    def update(self, i, delta):
        """Add delta to position i (1-indexed). O(log n)."""
        while i <= self.n:
            self.tree[i] += delta
            i += i & (-i)            # walk to parent: add lowest set bit

    def query(self, i):
        """Prefix sum from 1 to i inclusive (1-indexed). O(log n)."""
        total = 0
        while i > 0:
            total += self.tree[i]
            i -= i & (-i)            # strip lowest set bit → previous covered range
        return total

    def range_query(self, l, r):
        """Sum from l to r inclusive (1-indexed). O(log n)."""
        return self.query(r) - self.query(l - 1)

# Slow motion on nums=[1,3,5,7,9,11], BIT size=6:
# build by inserting each element:
#   update(1,1): bit[1]+=1, bit[2]+=1, bit[4]+=1
#   update(2,3): bit[2]+=3, bit[4]+=3
#   update(3,5): bit[3]+=5, bit[4]+=5
#   update(4,7): bit[4]+=7
#   update(5,9): bit[5]+=9, bit[6]+=9
#   update(6,11): bit[6]+=11
# query(6): bit[6]+bit[4] = (9+11)+(1+3+5+7) = 20+16 = 36 ✓

def test_harness(bit_class):
    nums = [1, 3, 5, 7, 9, 11]
    bit = bit_class(len(nums))
    for i, v in enumerate(nums):
        bit.update(i + 1, v)         # BIT uses 1-indexed positions

    tests = [
        (bit.query(6), 36),          # full prefix sum
        (bit.query(3), 9),           # prefix 1..3 = 1+3+5
        (bit.range_query(2, 4), 15), # range 2..4 = 3+5+7
        (bit.range_query(4, 6), 27), # range 4..6 = 7+9+11
    ]
    passed = 0
    for got, expected in tests:
        ok = got == expected
        if not ok:
            print(f"FAILED | expected={expected} | got={got}")
        passed += ok

    # update test: change index 2 (1-indexed) from 3 to 10 → delta=+7
    bit.update(2, 7)
    got = bit.query(6)               # should be 36+7=43
    ok = got == 43
    if not ok:
        print(f"FAILED | after update | expected=43 | got={got}")
    passed += ok
    print(f"{passed}/{len(tests)+1} tests passed")

test_harness(BITFull)
print("BITFull defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Range Sum Query Mutable — LC 307

---

```
PROBLEM:  Implement NumArray that supports:
          - update(i, val): set nums[i] = val
          - sumRange(l, r): return sum of nums[l..r]
APPROACH: Use BIT. Store current nums[] separately to compute delta on update.
          delta = new_val - old_val → update BIT by delta (not absolute value).

SLOW MOTION TRACE on nums = [1, 3, 5]:

  __init__: BIT(3), update(1,1), update(2,3), update(3,5)
    bit.tree = [0, 1, 4, 5]   (tree[2]=1+3=4, tree[3]=5)

  sumRange(0, 2):  bit.range_query(1, 3) = bit.query(3) - bit.query(0)
    bit.query(3): tree[3]=5, i=3-1=2, tree[2]=4, i=2-2=0 → 5+4=9 ✓

  update(1, 2):  old=nums[1]=3, delta=2-3=-1
    nums[1]=2
    bit.update(2, -1): tree[2]+=-1 → 3, tree[4] would +=-1 but n=3 so stop

  sumRange(0, 2):  now 1+2+5=8
    bit.query(3)=5+3=8 ✓

KEY INSIGHT: Store nums[] separately to compute delta = new - old.
             BIT stores cumulative sums — you add delta, not the absolute new value.
TIME:  O(n) init, O(log n) per update and sumRange
SPACE: O(n) — BIT array + nums copy
```

In [ ]:
class NumArray:
    """
    LC 307 — Range Sum Query - Mutable
    Approach: BIT for O(log n) update and range sum; track nums[] for delta computation.
    Time:  O(n) init, O(log n) update and sumRange
    Space: O(n) — BIT array + nums copy
    """
    def __init__(self, nums):
        self.n = len(nums)
        self.nums = nums[:]             # local copy to compute deltas on update
        self.bit = [0] * (self.n + 1)  # 1-indexed BIT
        for i, v in enumerate(nums):
            self._bit_update(i + 1, v) # build BIT by inserting each element

    def _bit_update(self, i, delta):
        while i <= self.n:
            self.bit[i] += delta
            i += i & (-i)              # propagate up using lowbit trick

    def _bit_query(self, i):
        total = 0
        while i > 0:
            total += self.bit[i]
            i -= i & (-i)              # strip lowbit to get previous range
        return total

    def update(self, index, val):
        """Set nums[index] = val. O(log n)."""
        delta = val - self.nums[index]  # how much to add to BIT — not absolute val
        self.nums[index] = val          # keep local copy in sync
        self._bit_update(index + 1, delta)  # BIT is 1-indexed

    def sumRange(self, left, right):
        """Sum of nums[left..right] inclusive, 0-indexed. O(log n)."""
        return self._bit_query(right + 1) - self._bit_query(left)

# Slow motion on nums=[1,3,5]:
# init: bit=[0,1,4,5] (tree[2]=1+3=4 since lowbit(2)=2 covers [1..2])
# sumRange(0,2): query(3)-query(0) = (bit[3]+bit[2]) - 0 = 5+4 = 9 ✓
# update(1,2): delta=2-3=-1, bit[2]-=1 → bit=[0,1,3,5]
# sumRange(0,2): query(3)-query(0) = 5+3 = 8 ✓ (1+2+5=8)

def test_harness(cls):
    tests = [
        ([1, 3, 5], 0, 2, None, 9),                  # sumRange(0,2)=9
        ([1, 3, 5], 1, None, 2, 8),                  # update(1,2) then sumRange(0,2)
        ([-2, 0, 3, -5, 2, -1], 0, 5, None, -3),    # negative values
        ([0, 0, 0], 0, 2, None, 0),                  # all zeros
    ]
    passed = 0
    for nums, l, r, update_val, expected in tests:
        na = cls(nums[:])
        if update_val is not None:
            na.update(l, update_val)   # update index l to update_val
            got = na.sumRange(0, len(nums)-1)
        else:
            got = na.sumRange(l, r)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | nums={nums} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(NumArray)
print("NumArray defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Count of Smaller Numbers After Self — LC 315

---

```
PROBLEM:  Given nums, return counts[i] = number of elements to the RIGHT of i
          that are smaller than nums[i].
APPROACH: Process from RIGHT to LEFT. For each element:
          1. Query BIT: how many already-inserted values are < nums[i]?
          2. Insert nums[i] into BIT.
          Coordinate compress values → BIT indexed by rank not raw value.

SLOW MOTION TRACE on nums = [5, 2, 6, 1]:

  sorted_unique = [1, 2, 5, 6]   rank: {1:1, 2:2, 5:3, 6:4}
  BIT size = 4, process right to left:

  i=3, val=1, rank=1:
    query(rank-1=0)=0  → counts[3]=0  (no elements to right yet)
    update(rank=1, +1)  BIT now has: 1 inserted

  i=2, val=6, rank=4:
    query(rank-1=3)=1  → counts[2]=1  (only 1 element < 6 seen so far: the 1)
    update(rank=4, +1)

  i=1, val=2, rank=2:
    query(rank-1=1)=1  → counts[1]=1  (elements < 2: just the 1)
    update(rank=2, +1)

  i=0, val=5, rank=3:
    query(rank-1=2)=2  → counts[0]=2  (elements < 5: 2 and 1)
    update(rank=3, +1)

  result (reversed): [2, 1, 1, 0] ✓

KEY INSIGHT: Process right-to-left so BIT contains only elements we've seen
             (= elements to the right of current position).
             query(rank-1) = count of elements with rank < current = elements < val.
TIME:  O(n log n) — n iterations, each O(log n) BIT operation
SPACE: O(n) — BIT + rank map
```

In [ ]:
def count_smaller(nums):
    """
    LC 315 — Count of Smaller Numbers After Self
    Approach: right-to-left scan with BIT + coordinate compression.
    Args:
        nums (List[int]): integer array.
    Returns:
        List[int]: counts[i] = # elements to the right of i smaller than nums[i].
    Time:  O(n log n) — n BIT operations, each O(log n)
    Space: O(n)       — BIT array + rank map
    """
    # coordinate compression: map each value to its rank [1..m]
    sorted_unique = sorted(set(nums))
    rank = {v: i + 1 for i, v in enumerate(sorted_unique)}
    m = len(sorted_unique)  # BIT size = number of distinct values

    bit = [0] * (m + 1)   # 1-indexed BIT

    def bit_update(i, delta):
        while i <= m:
            bit[i] += delta
            i += i & (-i)   # walk to next responsible node

    def bit_query(i):
        total = 0
        while i > 0:
            total += bit[i]
            i -= i & (-i)   # strip lowbit to get prefix sum
        return total

    result = []
    for val in reversed(nums):         # right to left — BIT holds only elements seen
        r = rank[val]
        result.append(bit_query(r - 1)) # count elements with rank < r = elements < val
        bit_update(r, 1)                # register this element

    return result[::-1]   # reverse because we processed right to left

# Slow motion on nums=[5,2,6,1]:
# rank={1:1, 2:2, 5:3, 6:4}
# process: 1→count=0,insert; 6→count=1,insert; 2→count=1,insert; 5→count=2,insert
# reversed result = [2,1,1,0]

def test_harness(fn):
    tests = [
        ([5, 2, 6, 1], [2, 1, 1, 0]),
        ([2, 0, 1], [2, 0, 0]),
        ([-1], [0]),
        ([-1, -1], [0, 0]),
        ([1, 2, 3], [0, 0, 0]),       # ascending — no smaller to the right
        ([3, 2, 1], [2, 1, 0]),       # descending — each is greater than all right
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(count_smaller)
print("count_smaller defined.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Count of Range Sum — LC 327

---

```
PROBLEM:  Given nums, lower, upper: count range sums sum(i,j) that lie in [lower,upper].
          sum(i,j) = nums[i] + ... + nums[j] for i <= j.
APPROACH: Prefix sums reduce the problem:
          sum(i,j) = prefix[j+1] - prefix[i]
          We need: lower <= prefix[j+1] - prefix[i] <= upper
          i.e.:  prefix[j+1]-upper <= prefix[i] <= prefix[j+1]-lower

          Use MERGE SORT divide-and-conquer on prefix array:
          During merge of two sorted halves, for each prefix[j] in right half,
          count prefix[i] in left half with value in [prefix[j]-upper, prefix[j]-lower].
          Both halves are sorted → two pointers lo,hi walk left half without backtrack.

SLOW MOTION TRACE on nums=[-2,5,-1], lower=-2, upper=2:

  prefix = [0, -2, 3, 2]
  All range sums:
    sum(0,0)=-2, sum(0,1)=3, sum(0,2)=2
    sum(1,1)=5,  sum(1,2)=4
    sum(2,2)=-1
  In [-2..2]: {-2, 2, -1} → 3 pairs

  merge_sort on prefix=[0,-2,3,2]:
    merge_sort([0,-2]): produces [-2,0], counts cross pairs from [{0},{-2}]
      left=[0], right=[-2]
      for right_val=-2: want left in [-2-2,-2-(-2)]=[-4,0] → left[0]=0 ∈[-4,0] → count+=1
    merge_sort([3,2]): produces [2,3], counts cross pairs
      left=[3], right=[2]
      for right_val=2: want left in [2-2,2-(-2)]=[0,4] → left[3]=3 ∈[0,4] → count+=1
    merge left=[-2,0] right=[2,3]:
      for right_val=2: want left in [0,4] → left in {-2,0}: only 0∈[0,4] → count+=1
      for right_val=3: want left in [1,5] → left in {-2,0}: none → count+=0
  total count=3 ✓

KEY INSIGHT: Sorting prefix values enables two-pointer counting of valid left values
             for each right value — no re-scanning needed because both sides are sorted.
TIME:  O(n log n) — merge sort on prefix array of length n+1
SPACE: O(n)       — recursion stack + temporary arrays
```

In [ ]:
def count_range_sum(nums, lower, upper):
    """
    LC 327 — Count of Range Sum
    Approach: prefix sums + merge sort; count cross-half pairs during merge.
    Args:
        nums (List[int]): integer array.
        lower (int): lower bound of valid range sum.
        upper (int): upper bound of valid range sum.
    Returns:
        int: number of range sums in [lower, upper].
    Time:  O(n log n) — merge sort on prefix array of length n+1
    Space: O(n)       — recursion depth + temporary merge arrays
    """
    # build prefix sum array: prefix[i] = sum of nums[0..i-1]
    prefix = [0]
    for x in nums:
        prefix.append(prefix[-1] + x)

    count = [0]   # mutable container so inner function can modify it

    def merge_count(arr):
        if len(arr) <= 1:
            return arr           # base case: already sorted

        mid = len(arr) // 2
        left = merge_count(arr[:mid])    # sort left half, count pairs within
        right = merge_count(arr[mid:])   # sort right half, count pairs within

        # count cross-half pairs: left[i] from left half, right[j] from right half
        # valid if lower <= right[j] - left[i] <= upper
        # ↔  right[j] - upper <= left[i] <= right[j] - lower
        lo = hi = 0   # two pointers into left half
        for rval in right:
            # advance lo: first left[lo] >= rval - upper
            while lo < len(left) and left[lo] < rval - upper:
                lo += 1
            # advance hi: first left[hi] > rval - lower
            while hi < len(left) and left[hi] <= rval - lower:
                hi += 1
            count[0] += hi - lo   # all left values in [lo..hi) are valid for rval

        # standard merge to produce sorted array
        merged = []
        i = j = 0
        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                merged.append(left[i]); i += 1
            else:
                merged.append(right[j]); j += 1
        merged.extend(left[i:])
        merged.extend(right[j:])
        return merged

    merge_count(prefix)
    return count[0]

# Slow motion on nums=[-2,5,-1], lower=-2, upper=2:
# prefix=[0,-2,3,2]
# cross pairs found:
#   left=[-2], right=[0]: rval=0, want left in [-2,2] → -2 qualifies → count=1
#   left=[2],  right=[3]: rval=3, want left in [1,5]  → 2∈[1,5] → count=2
#   final merge: left=[-2,0] vs right=[2,3]
#     rval=2: want left in [0,4] → 0∈[0,4] → count=3
#     rval=3: want left in [1,5] → neither -2 nor 0 in [1,5] → count=3

def test_harness(fn):
    tests = [
        ([-2, 5, -1], -2, 2, 3),
        ([0], 0, 0, 1),
        ([0], -1, 0, 1),
        ([1, 2, 3], 3, 5, 3),    # sums: 1,2,3,3,5,6 → 3,3,5 in [3,5]
        ([], 0, 0, 0),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(count_range_sum)
print("count_range_sum defined.")

<a id='10'></a>
## 10. 🗺️ The Segment Tree / BIT Decision Map

```
QUESTION TYPE                          KEY TECHNIQUE              LC PROBLEMS
────────────────────────────────────────────────────────────────────────────────
Range sum + point update               BIT (simpler to code)      307
Range min/max + point update           Segment Tree               (no BIT for min/max)
Range sum + range update               Seg Tree + lazy prop       (beyond scope here)
Count elements < x seen so far         BIT + coord compression    315
Count range sums in [lo, hi]           Prefix + merge sort        327
Inversions in array                    BIT right-to-left          (LC 315 variant)
────────────────────────────────────────────────────────────────────────────────

COORDINATE COMPRESSION RECIPE:
  sorted_unique = sorted(set(nums))         # deduplicated sorted values
  rank = {v: i+1 for i, v in enumerate(sorted_unique)}  # 1-indexed ranks
  BIT(len(sorted_unique))                   # BIT size = distinct count

BIT QUERY PATTERNS:
  bit_query(r - 1)    →  count of elements STRICTLY LESS THAN value with rank r
  bit_query(r)        →  count of elements LESS THAN OR EQUAL TO rank r
  bit_query(r2) - bit_query(r1-1)  →  count in rank range [r1..r2]

MERGE SORT COUNT RECIPE:
  1. Build prefix sum array
  2. In merge step: left sorted, right sorted → two pointers count valid cross pairs
  3. lo, hi advance monotonically → O(n) cross-pair counting per merge level
  Total: O(n log n)
```

<a id='11'></a>
## 11. 📋 Interview Cheat Sheet

### When to reach for Segment Tree / BIT:

| Signal | What to Do |
|--------|------------|
| "range sum" + updates possible | BIT |
| "range min/max" + updates | Segment Tree |
| "how many elements < x seen so far" | BIT + coord compression |
| "count range sums in [lo, hi]" | Prefix sums + merge sort |
| "inversions" in array | BIT right-to-left |

### The O(log n) operations — memorize these:

```python
# BIT update: add delta to position i (1-indexed)
while i <= n: bit[i] += delta; i += i & (-i)

# BIT query: prefix sum 1..i
total = 0
while i > 0: total += bit[i]; i -= i & (-i)

# lowbit: lowest set bit
i & (-i)
```

### Common templates:

```python
# TEMPLATE 1: BIT — point update, prefix query
bit = [0] * (n + 1)       # 1-indexed
def update(i, d):          # add d at position i
    while i <= n: bit[i] += d; i += i & (-i)
def query(i):              # prefix sum 1..i
    s = 0
    while i > 0: s += bit[i]; i -= i & (-i)
    return s

# TEMPLATE 2: COORDINATE COMPRESSION
vals = sorted(set(nums))
rank = {v: i+1 for i, v in enumerate(vals)}  # 1-indexed
# query(rank[x] - 1) → count elements strictly less than x

# TEMPLATE 3: SEGMENT TREE (recursive, 1-indexed node array)
tree = [0] * (4 * n)
def build(node, l, r, nums):
    if l==r: tree[node]=nums[l]; return
    mid=(l+r)//2
    build(2*node,l,mid,nums); build(2*node+1,mid+1,r,nums)
    tree[node]=tree[2*node]+tree[2*node+1]
def query(node,l,r,ql,qr):
    if ql<=l and r<=qr: return tree[node]  # full overlap
    if qr<l or r<ql: return 0              # no overlap
    mid=(l+r)//2
    return query(2*node,l,mid,ql,qr)+query(2*node+1,mid+1,r,ql,qr)
```

### Gotchas to not forget:

```
❌  BIT with i=0 — lowbit(0)=0 causes infinite loop; always use 1-indexed
✅  Shift all indices by +1 before BIT operations
❌  BIT update with absolute value on mutable array — always update with delta
✅  delta = new_val - old_val; then bit_update(i, delta)
❌  Segment tree with tree[2*n] — needs 4*n nodes for non-power-of-2 n
✅  Allocate tree[4*n] to be safe
❌  Merge sort count: forget to advance both lo and hi for each right value
✅  Both lo and hi only advance (never reset) — total O(n) per merge level
```

<a id='12'></a>
## 12. 🗺️ Summary Map

```
              RANGE QUERY STRUCTURES
                        │
           ┌────────────┼───────────────┐
           │            │               │
        SEGMENT       BIT /          MERGE
         TREE        FENWICK          SORT
           │            │               │
       4n array      flat n+1       divide &
       recursive     1-indexed      conquer on
       build O(n)    lowbit trick   prefix array
           │            │               │
       range min    point update    count pairs
       range max    prefix sum      in valid range
       lazy update  coord compress  LC 327
           │            │
         LC 307     LC 307, 315

BIT LOWBIT TRICK:
  i & (-i)  =  lowest set bit  =  range width

  Update i → walk UP:   i += i&(-i)   (expand responsibility)
  Query  i → walk DOWN: i -= i&(-i)   (peel off covered ranges)

COORDINATE COMPRESSION:
  value space [−10^9 .. 10^9]  →  rank space [1..m]
  query(rank-1) = count of elements STRICTLY LESS THAN current
```

---
*End of Segment Tree / BIT Master Guide — Sean Edition*